# Real-World AI-Driven Analytics Solution Using Python
## Deliverable: Data Preprocessing and Initial Model Development
**Dataset:** Heart Disease (UCI Cleveland) &nbsp;|&nbsp; **Task:** Binary classification — predict presence of heart disease

This notebook explores, cleans and transforms the dataset, then builds and
evaluates initial models (Logistic Regression and Random Forest). It is
structured to run end-to-end in Google Colab with no manual upload — the data
is read directly from the UCI repository.

**Team:** *<add member names and roles here>*

## 1. Setup and Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)

sns.set_theme(style='whitegrid', palette='deep')
RANDOM_STATE = 42

## 2. Accessing the Data

The processed Cleveland file has no header row, so we assign the 14 official
column names. Missing values are encoded as `?` in the source file.

In [ ]:
URL = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
       'heart-disease/processed.cleveland.data')

columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

df = pd.read_csv(URL, header=None, names=columns, na_values='?')
df.head()

**Column reference**

| Feature | Meaning |
|---|---|
| age | Age in years |
| sex | 1 = male, 0 = female |
| cp | Chest pain type (1=typical angina … 4=asymptomatic) |
| trestbps | Resting blood pressure (mm Hg) |
| chol | Serum cholesterol (mg/dl) |
| fbs | Fasting blood sugar > 120 mg/dl (1/0) |
| restecg | Resting ECG result (0,1,2) |
| thalach | Maximum heart rate achieved |
| exang | Exercise-induced angina (1/0) |
| oldpeak | ST depression induced by exercise |
| slope | Slope of peak exercise ST segment (1,2,3) |
| ca | Number of major vessels (0–3) coloured by fluoroscopy |
| thal | 3=normal, 6=fixed defect, 7=reversible defect |
| num | Diagnosis (0 = no disease; 1–4 = disease present) — **target** |

## 3. Data Exploration

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
# Missing values (ca and thal contain a handful of '?')
df.isna().sum()

In [ ]:
# Binarise the target: 0 = no disease, 1 = disease present
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['num'])

plt.figure(figsize=(5, 3.4))
ax = sns.countplot(data=df, x='target')
ax.set_xticklabels(['No disease (0)', 'Disease (1)'])
ax.set_title('Target distribution'); ax.set_xlabel(''); ax.set_ylabel('Patients')
plt.tight_layout(); plt.show()
df['target'].value_counts(normalize=True).round(3)

In [ ]:
# Correlation with the target
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Feature correlation matrix'); plt.tight_layout(); plt.show()

In [ ]:
# Key clinical features vs target
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
sns.boxplot(data=df, x='target', y='thalach', ax=axes[0,0]); axes[0,0].set_title('Max heart rate')
sns.boxplot(data=df, x='target', y='oldpeak', ax=axes[0,1]); axes[0,1].set_title('ST depression')
sns.histplot(data=df, x='age', hue='target', bins=20, kde=True, ax=axes[1,0]); axes[1,0].set_title('Age')
sns.countplot(data=df, x='cp', hue='target', ax=axes[1,1]); axes[1,1].set_title('Chest pain type')
plt.tight_layout(); plt.show()

## 4. Data Cleaning and Preprocessing

Steps: drop duplicates, impute the few missing `ca`/`thal` values with the
median, one-hot encode the nominal categorical columns, then scale the
numeric features for the linear model.

In [ ]:
# 4a. Duplicates
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows')

# 4b. Impute missing values (median is robust to outliers)
for col in ['ca', 'thal']:
    df[col] = df[col].fillna(df[col].median())
print('Remaining missing values:', int(df.isna().sum().sum()))

In [ ]:
# 4c. One-hot encode nominal categoricals (cp, restecg, slope, thal)
categorical = ['cp', 'restecg', 'slope', 'thal']
df = pd.get_dummies(df, columns=categorical, drop_first=True)
df.shape

In [ ]:
# 4d. Train/test split (stratified to preserve class balance) then scale
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on TRAIN only
X_test_scaled = scaler.transform(X_test)         # avoid data leakage
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. Initial Model Development

Two baselines: Logistic Regression (interpretable linear model, uses scaled
features) and Random Forest (non-linear ensemble, scale-invariant).

In [ ]:
def evaluate(model, X_tr, X_te, name):
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    prob = model.predict_proba(X_te)[:, 1]
    print(f'=== {name} ===')
    print('Accuracy :', round(accuracy_score(y_test, pred), 3))
    print('ROC-AUC  :', round(roc_auc_score(y_test, prob), 3))
    print(classification_report(y_test, pred))
    return model, pred, prob

logreg, lr_pred, lr_prob = evaluate(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    X_train_scaled, X_test_scaled, 'Logistic Regression')

rf, rf_pred, rf_prob = evaluate(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    X_train, X_test, 'Random Forest')   # trees do not need scaling

In [ ]:
# Confusion matrix for the stronger baseline
plt.figure(figsize=(4.5, 3.6))
cm = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No disease', 'Disease'], yticklabels=['No disease', 'Disease'])
plt.title('Confusion matrix — Logistic Regression')
plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.tight_layout(); plt.show()

In [ ]:
# ROC curves
plt.figure(figsize=(5, 4))
for prob, name in [(lr_prob, 'Logistic Regression'), (rf_prob, 'Random Forest')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, prob):.3f})')
plt.plot([0, 1], [0, 1], '--', color='grey')
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('ROC curves'); plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

## 6. Hyperparameter Tuning

5-fold grid search over the Random Forest, optimising ROC-AUC.

In [ ]:
param_grid = {'n_estimators': [100, 200],
              'max_depth': [5, 10, None],
              'min_samples_split': [2, 5]}

grid = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE),
                    param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print('Best CV ROC-AUC:', round(grid.best_score_, 3))

best_rf = grid.best_estimator_
tuned_pred = best_rf.predict(X_test)
tuned_prob = best_rf.predict_proba(X_test)[:, 1]
print('Tuned test accuracy:', round(accuracy_score(y_test, tuned_pred), 3))
print('Tuned test ROC-AUC :', round(roc_auc_score(y_test, tuned_prob), 3))

In [ ]:
# Feature importance from the tuned model
imp = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(7, 4.5))
imp.tail(10).plot(kind='barh')
plt.title('Top 10 feature importances (tuned Random Forest)')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()

## 7. Ethical Considerations

- **Bias and representativeness.** The Cleveland cohort skews male and older;
  performance may not transfer to women, younger patients or other
  populations. Subgroup metrics (e.g. accuracy by `sex`) should be checked
  before any real use.
- **Cost of errors.** A false negative (missing true disease) is clinically
  more dangerous than a false positive. Recall on the positive class and the
  classification threshold matter more than raw accuracy.
- **Transparency.** Logistic-regression coefficients and tree feature
  importances make the drivers inspectable, supporting clinician trust.
- **Decision support, not replacement.** This is an initial model on a small
  dataset and must not be used for autonomous diagnosis.
- **Privacy.** Even with de-identified data, health information must be
  handled under appropriate governance.

In [ ]:
# Example fairness check: accuracy by sex
test_df = X_test.copy()
test_df['actual'] = y_test.values
test_df['pred'] = lr_pred
for s, label in [(1, 'Male'), (0, 'Female')]:
    sub = test_df[test_df['sex'] == s]
    if len(sub):
        acc = accuracy_score(sub['actual'], sub['pred'])
        print(f'{label:7s} (n={len(sub):2d}): accuracy = {acc:.3f}')